# STAR-RIS RSMA Kaggle Worker

This notebook clones the source repository, runs one compact experiment job, and exports only the files needed for download. Edit the first code cell before saving a Kaggle version.


In [ ]:
# ===== Job configuration =====
REPO_URL = 'https://github.com/Juliolayme/STAR_RIS_RSMA_TD3.git'
BRANCH = 'main'

# JOB_KIND: 'learned', 'solver', or 'postprocess'
JOB_KIND = 'learned'

# Scalability setting
N_RIS = 32  # one of: 16, 32, 64, 96, 128

# Learned job settings
METHOD = 'td3'  # 'td3', 'ddpg', or 'ppo'
SEED = 0
RUN_TD3_EXTRAS = True  # ablation + latency only when METHOD == 'td3'
KEEP_LATEST_CHECKPOINT = False  # keep best.pt; drop latest.pt to reduce output size

# Solver job settings
SOLVER_METHOD = 'ao_sca'  # 'ao_sca', 'ao_grid', or 'analytical_ris'
SHARD_START = 0
SHARD_COUNT = 100
SOLVER_SEED = 10000

# Locked ScenarioBank sizes from README
TRAIN_COUNT = 10000
VALIDATION_COUNT = 1000
TEST_COUNT = 1000

# Latency settings for TD3 extras
LATENCY_WARMUP = 20
LATENCY_COUNT = 500


In [ ]:
from pathlib import Path
import os, shutil, subprocess, textwrap, zipfile

WORK = Path('/kaggle/working')
REPO = WORK / 'STAR_RIS_RSMA_TD3'
CONFIG = f'configs/siso_n{N_RIS}.yaml'
BANK_DIR = 'artifacts/scenario_banks'
TEST_BANK = f'{BANK_DIR}/N{N_RIS}_test.npz'

def run(cmd, cwd=REPO):
    print('\n$ ' + cmd)
    subprocess.run(cmd, shell=True, cwd=cwd, check=True)

if REPO.exists():
    shutil.rmtree(REPO)
run(f'git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}', cwd=WORK)

# Kaggle normally already has a GPU-enabled torch. Install the project without
# forcing a torch wheel replacement, then install only non-torch dependencies.
run('python -m pip install -q -e . --no-deps')
run('python -m pip install -q "numpy>=1.26" "pandas>=2.1" "pyyaml>=6.0" "scipy>=1.11" "matplotlib>=3.8"')
run('python - <<\'PY\'\nimport torch\nprint(\'torch\', torch.__version__, \'cuda\', torch.cuda.is_available())\nPY')


In [ ]:
# Create locked ScenarioBanks deterministically inside this Kaggle run.
# They are removed from final output after the job; the small manifest is kept.
run(
    f'python scripts/create_scenario_banks.py --config {CONFIG} '
    f'--output-dir {BANK_DIR} --train-count {TRAIN_COUNT} '
    f'--validation-count {VALIDATION_COUNT} --test-count {TEST_COUNT}'
)


In [ ]:
if JOB_KIND == 'learned':
    train_dir = f'results/train/{METHOD}/N{N_RIS}/seed_{SEED}'
    test_csv = f'results/test/{METHOD}/N{N_RIS}/seed_{SEED}.csv'
    run(f'python scripts/run_train.py --method {METHOD} --config {CONFIG} --seed {SEED} --output {train_dir}')
    run(f'python scripts/run_evaluate.py --method {METHOD} --config {CONFIG} --checkpoint {train_dir}/best.pt --bank {TEST_BANK} --seed {SEED} --output {test_csv}')
    if METHOD == 'td3' and RUN_TD3_EXTRAS:
        run(f'python scripts/run_ablation.py --method td3 --config {CONFIG} --checkpoint {train_dir}/best.pt --bank {TEST_BANK} --seed {SEED} --output results/ablations/N{N_RIS}/seed_{SEED}.csv')
        run(f'python scripts/benchmark_latency.py --method td3 --config {CONFIG} --checkpoint {train_dir}/best.pt --bank {TEST_BANK} --warmup {LATENCY_WARMUP} --count {LATENCY_COUNT} --output results/latency/td3_N{N_RIS}_seed{SEED}.csv')
    if not KEEP_LATEST_CHECKPOINT:
        latest = REPO / train_dir / 'latest.pt'
        if latest.exists():
            latest.unlink()
elif JOB_KIND == 'solver':
    out = f'results/solvers/N{N_RIS}/{SOLVER_METHOD}_{SHARD_START}_{SHARD_START + SHARD_COUNT}.csv'
    run(f'python scripts/run_solver.py --method {SOLVER_METHOD} --config {CONFIG} --bank {TEST_BANK} --seed {SOLVER_SEED} --start {SHARD_START} --count {SHARD_COUNT} --output {out}')
elif JOB_KIND == 'postprocess':
    run(f'python scripts/merge_results.py --inputs results/test/**/*.csv results/solvers/**/*.csv --output results/merged/N{N_RIS}_all.csv')
    run(f'python scripts/analyze_results.py --inputs results/merged/N{N_RIS}_all.csv --output-dir results/statistics/N{N_RIS}')
    run(f'python scripts/plot_results.py --inputs results/merged/N{N_RIS}_all.csv --output results/figures/N{N_RIS}_sum_rate.png')
else:
    raise ValueError(f'Unknown JOB_KIND: {JOB_KIND}')


In [ ]:
# Compact export: keep results and small bank manifest, drop large NPZ banks and source clone.
EXPORT = WORK / 'export'
if EXPORT.exists():
    shutil.rmtree(EXPORT)
EXPORT.mkdir(parents=True)

if (REPO / 'results').exists():
    shutil.copytree(REPO / 'results', EXPORT / 'results')
manifest = REPO / BANK_DIR / f'N{N_RIS}_manifest.json'
if manifest.exists():
    target = EXPORT / 'artifacts' / 'scenario_banks'
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(manifest, target / manifest.name)

label = f'{JOB_KIND}_N{N_RIS}'
if JOB_KIND == 'learned':
    label += f'_{METHOD}_seed{SEED}'
elif JOB_KIND == 'solver':
    label += f'_{SOLVER_METHOD}_{SHARD_START}_{SHARD_START + SHARD_COUNT}'
archive = WORK / f'star_ris_rsma_{label}'
zip_path = shutil.make_archive(str(archive), 'zip', EXPORT)

shutil.rmtree(EXPORT)
shutil.rmtree(REPO)
print(f'Created compact output: {zip_path}')
print('Kaggle output should only need this zip file.')
